In [3]:
import polars as pl

# Just creating a fake DataFrame to experiment with
training_data = pl.DataFrame({'age': [22, 32, 23, 38, 32, 55, 40], 'salary': [120000, 100000, 80000, 45000, 90000, 75000, 55000]})
display(training_data)
test_data = pl.DataFrame({'age': [33, 42, 25], 'salary': [115000, 143000, 53000]})
test_data

age,salary
i64,i64
22,120000
32,100000
23,80000
38,45000
32,90000
55,75000
40,55000


age,salary
i64,i64
33,115000
42,143000
25,53000


## Using Scikit-Learn

In [6]:
# If we need to do some kind of distance-based algorithm with these columns, it would
# help to transform them to the same scale first.
#
# Min-max scaling will make the values span the range 0.0 - 1.0
#
# We could write a native expression for it, but SciKit-Learn has
# a built-in way to do this for us:
from sklearn.preprocessing import MinMaxScaler

# Create an instance of the transform object we're going to use
scaler = MinMaxScaler()

scaled_train = scaler.fit_transform(training_data)
scaled_test = scaler.transform(test_data)
print("Training")
display(pl.DataFrame(scaled_train, schema=training_data.columns))
print("\nTest")
display(pl.DataFrame(scaled_test, schema=test_data.columns))

Training


age,salary
f64,f64
0.0,1.0
0.30303,0.733333
0.030303,0.466667
0.484848,0.0
0.30303,0.6
1.0,0.4
0.545455,0.133333



Test


age,salary
f64,f64
0.333333,0.933333
0.606061,1.306667
0.090909,0.106667


## Native Polars

In [10]:
# A Min-Max scaler is given by this formula:
#
# (value - min)
# -------------
#  (max - min)
#
# We can use polars native expressions to create columns with their normalized
# counterpart

columns_to_scale = ['age', 'salary']
train_mins = {}
train_maxs = {}
for column in columns_to_scale:
  train_mins[column] = training_data.select(pl.col(column)).to_series().min()
  train_maxs[column] = training_data.select(pl.col(column)).to_series().max()

training_scaled = training_data
testing_scaled = test_data

for column in columns_to_scale:
  training_scaled = training_scaled.with_columns(((pl.col(column) - train_mins[column]) / (train_maxs[column] - train_mins[column])).name.suffix('_scaled'))
  testing_scaled = testing_scaled.with_columns(((pl.col(column) - train_mins[column]) / (train_maxs[column] - train_mins[column])).name.suffix('_scaled'))

print("Training")
display(training_scaled)
print("\nTest")
display(testing_scaled)

Training


age,salary,age_scaled,salary_scaled
i64,i64,f64,f64
22,120000,0.0,1.0
32,100000,0.30303,0.733333
23,80000,0.030303,0.466667
38,45000,0.484848,0.0
32,90000,0.30303,0.6
55,75000,1.0,0.4
40,55000,0.545455,0.133333



Test


age,salary,age_scaled,salary_scaled
i64,i64,f64,f64
33,115000,0.333333,0.933333
42,143000,0.606061,1.306667
25,53000,0.090909,0.106667
